# DermaVQA - LoRA/QLoRA VLM sobre `dataset_enriched`

Entregable de Santino para fine-tunear el VLM definido en el survey: **`Qwen/Qwen2.5-VL-3B-Instruct`** con **imagen + `question_es`** y target **`synthesized_answer_es` si existe, si no `answer_es`** del dataset enriquecido.

El notebook esta preparado para Google Cloud, Colab, Kaggle o ejecucion local. Por defecto **no descarga modelos ni entrena**: las partes pesadas quedan detras de flags (`RUN_TRAINING`, `RUN_EVAL_GENERATION`, `PACKAGE_RESULTS`).

Para Google Cloud, la configuracion recomendada es una VM con **NVIDIA L4 24GB** y estas variables opcionales:

```bash
export DERMAVQA_VLM_MODEL="Qwen/Qwen2.5-VL-3B-Instruct"
export DERMAVQA_IMAGE_ROOT="/mnt/disks/dermavqa/images_final"
export DERMAVQA_VLM_LORA_OUTPUT_DIR="$PWD/outputs/results/dataset_enriched/vlm_lora"
```

Outputs esperados en `outputs/results/dataset_enriched/vlm_lora/`:

- `predictions_valid.csv`
- `predictions_test.csv`
- `metrics_valid.csv`
- `metrics_test.csv`
- `manual_review_20.csv`
- `dataset_enriched_vlm_lora_adapter_results.zip`


## 0. Setup

Ejecutar una vez en Colab/Kaggle. Si corrés local y ya tenés dependencias, podés saltear esta celda.


In [ ]:
# Puede pedir reiniciar kernel despues de instalar.
%pip install -q 'pandas>=2.0' 'numpy>=1.24' 'pillow>=10.0' 'tqdm>=4.66' 'sacrebleu>=2.4' 'rouge-score>=0.1.2' 'transformers>=4.51.0' 'accelerate>=1.2.0' 'peft>=0.13.0' 'bitsandbytes>=0.43.0' 'datasets>=3.0.0' 'qwen-vl-utils>=0.0.8'

# Opcional, mas pesado:
# %pip install -q bert-score


In [ ]:
import itertools
import json
import os
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'outputs' / 'datasets').exists() or ((candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists()):
            return candidate
    return start


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = Path(os.environ.get('DERMAVQA_VLM_LORA_OUTPUT_DIR', PROJECT_ROOT / 'outputs' / 'results' / 'dataset_enriched' / 'vlm_lora')).expanduser().resolve()
RAW_DIR = OUTPUT_DIR / 'raw_dataset'
ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATASET_VARIANT = 'dataset_enriched'
METHOD = 'vlm_lora'
DATASET_ZIP_NAME = 'dermavqa_iiyi_llm_synthesized_answer_finetune.zip'
DATASET_JSONL_NAME = 'dermavqa_iiyi_llm_synthesized_answer_finetune.jsonl'
DATASET_CSV_NAME = 'dermavqa_iiyi_llm_synthesized_answer_finetune.csv'

EXPECTED_SPLIT_COUNTS = {'train': 2473, 'valid': 157, 'test': 314}
EXPECTED_CASE_COUNTS = {'train': 842, 'valid': 56, 'test': 100}
STRICT_EXPECTED_COUNTS = True

MODEL_ID = os.environ.get('DERMAVQA_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct')
USE_QLORA = True
RUN_TRAINING = False
RUN_EVAL_GENERATION = False
RUN_BERTSCORE = False
PACKAGE_RESULTS = False

TRAIN_SMOKE_TEST = True
SMOKE_MAX_TRAIN_SAMPLES = 32
SMOKE_MAX_EVAL_SAMPLES = 24
SMOKE_MAX_STEPS = 3
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
MAX_TEXT_TOKENS = 1024
MAX_NEW_TOKENS = 128

# Qwen2.5-VL usa patches de 28x28. Este limite baja memoria sin eliminar la senal visual.
MIN_IMAGE_PIXELS = 128 * 28 * 28
MAX_IMAGE_PIXELS = 256 * 28 * 28

# En Google Cloud/Colab/Kaggle el zip no trae imagenes: apuntar a images_final si las rutas absolutas no existen.
IMAGE_ROOT_OVERRIDE = os.environ.get('DERMAVQA_IMAGE_ROOT', '').strip() or None
SEARCH_IMAGE_ROOTS = True

print('Python:', sys.version)
print('Project root:', PROJECT_ROOT)
print('Output dir:', OUTPUT_DIR)
print(json.dumps({'dataset_variant': DATASET_VARIANT, 'method': METHOD, 'model_id': MODEL_ID, 'use_qlora': USE_QLORA, 'run_training': RUN_TRAINING, 'run_eval_generation': RUN_EVAL_GENERATION, 'package_results': PACKAGE_RESULTS, 'image_root_override': IMAGE_ROOT_OVERRIDE, 'max_image_pixels': MAX_IMAGE_PIXELS}, indent=2, ensure_ascii=False))


## 1. Cargar el zip del dataset enriquecido

Busca `outputs/datasets/dermavqa_iiyi_llm_synthesized_answer_finetune.zip`. En Colab permite subirlo si no está disponible.


In [ ]:
def candidate_search_roots() -> list[Path]:
    roots = [PROJECT_ROOT, Path.cwd(), Path('/content'), Path('/kaggle/input')]
    unique = []
    for root in roots:
        try:
            resolved = root.expanduser().resolve()
        except Exception:
            continue
        if resolved.exists() and resolved not in unique:
            unique.append(resolved)
    return unique


def find_file_by_name(file_name: str) -> Path | None:
    for candidate in [PROJECT_ROOT / 'outputs' / 'datasets' / file_name, Path.cwd() / file_name, OUTPUT_DIR / file_name]:
        if candidate.exists():
            return candidate.resolve()
    for root in candidate_search_roots():
        try:
            matches = list(root.rglob(file_name))
        except Exception:
            matches = []
        if matches:
            return matches[0].resolve()
    return None


def upload_zip_in_colab(file_name: str) -> Path | None:
    try:
        from google.colab import files  # type: ignore
    except Exception:
        return None
    print('No encontré ' + file_name + '. Subilo desde tu máquina.')
    uploaded = files.upload()
    if file_name not in uploaded:
        return None
    return (Path.cwd() / file_name).resolve()


dataset_zip_path = find_file_by_name(DATASET_ZIP_NAME) or upload_zip_in_colab(DATASET_ZIP_NAME)
if dataset_zip_path is None or not dataset_zip_path.exists():
    raise FileNotFoundError('No se encontró ' + DATASET_ZIP_NAME + '. En local debe estar en outputs/datasets/; en Colab/Kaggle subilo o agregalo como input dataset.')

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(dataset_zip_path) as archive:
    archive.extractall(RAW_DIR)
    zip_entries = archive.namelist()

print('Zip:', dataset_zip_path)
print('Extraído en:', RAW_DIR)
for entry in zip_entries:
    print('-', entry)

jsonl_path = RAW_DIR / DATASET_JSONL_NAME
csv_path = RAW_DIR / DATASET_CSV_NAME
manifest_path = RAW_DIR / 'manifest.json'
if jsonl_path.exists():
    df = pd.read_json(jsonl_path, lines=True)
elif csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError('El zip no contiene JSONL ni CSV esperado.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
TARGET_COLUMN = 'synthesized_answer_es' if 'synthesized_answer_es' in df.columns else 'answer_es'
df['target_answer_es'] = df[TARGET_COLUMN]

print('Manifest rows:', manifest.get('row_count_by_image'))
print('Filas:', len(df))
print('Columnas:', list(df.columns))
print('Target usado:', TARGET_COLUMN)
try:
    display(df.head(3))
except NameError:
    print(df.head(3).to_string())


## 2. Validaciones de splits, conteos e imágenes

El dataset está expandido por imagen. Para VLM se mantiene cada fila por imagen, y se valida que no haya leakage de `encounter_id` entre splits.


In [ ]:
required_columns = {'split', 'encounter_id', 'image_id', 'image_path', 'question_es', TARGET_COLUMN, 'target_answer_es'}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError('Faltan columnas requeridas: ' + str(missing_columns))

split_counts = df['split'].value_counts().to_dict()
case_counts = df.groupby('split')['encounter_id'].nunique().to_dict()
empty_questions = int(df['question_es'].fillna('').astype(str).str.strip().eq('').sum())
empty_targets = int(df['target_answer_es'].fillna('').astype(str).str.strip().eq('').sum())
duplicate_rows = int(df.duplicated(['split', 'encounter_id', 'image_id']).sum())

print('Filas por split:', split_counts)
print('Casos por split:', case_counts)
print('Preguntas vacías:', empty_questions)
print('Targets vacíos:', empty_targets)
print('Duplicados split/encounter/image:', duplicate_rows)

if STRICT_EXPECTED_COUNTS and split_counts != EXPECTED_SPLIT_COUNTS:
    raise AssertionError('Conteos por split inesperados: ' + str(split_counts))
if STRICT_EXPECTED_COUNTS and case_counts != EXPECTED_CASE_COUNTS:
    raise AssertionError('Conteos por caso inesperados: ' + str(case_counts))
if empty_questions or empty_targets:
    raise AssertionError('Hay preguntas o targets vacíos.')
if duplicate_rows:
    raise AssertionError('Hay filas duplicadas por split/encounter_id/image_id.')

split_to_cases = {split: set(part['encounter_id']) for split, part in df.groupby('split')}
for left, right in itertools.combinations(sorted(split_to_cases), 2):
    overlap = split_to_cases[left] & split_to_cases[right]
    if overlap:
        raise AssertionError('Leakage entre ' + left + ' y ' + right + ': ' + str(len(overlap)))
print('OK: splits sin leakage por encounter_id.')


In [ ]:
def build_image_lookup(search_roots: list[Path], wanted_names: set[str]) -> dict[str, Path]:
    lookup = {}
    for root in search_roots:
        if not root.exists():
            continue
        print('Buscando imágenes en:', root)
        try:
            for path in root.rglob('*'):
                if path.is_file() and path.name in wanted_names and path.name not in lookup:
                    lookup[path.name] = path.resolve()
                    if len(lookup) == len(wanted_names):
                        return lookup
        except Exception as exc:
            print('No pude recorrer', root, '->', exc)
    return lookup


def resolve_image_paths(frame: pd.DataFrame) -> pd.Series:
    direct = []
    missing_names = set()
    for _, row in frame.iterrows():
        candidate = Path(str(row['image_path'])).expanduser()
        if candidate.exists():
            direct.append(str(candidate.resolve()))
        else:
            direct.append(None)
            missing_names.add(str(row['image_id']))
    direct_series = pd.Series(direct, index=frame.index, dtype='object')
    if not missing_names or not SEARCH_IMAGE_ROOTS:
        return direct_series
    roots = []
    if IMAGE_ROOT_OVERRIDE:
        roots.append(Path(IMAGE_ROOT_OVERRIDE).expanduser())
    roots.extend([PROJECT_ROOT / 'data' / 'iiyi' / 'images_final', PROJECT_ROOT / 'data', Path('/content'), Path('/kaggle/input')])
    roots = [root.resolve() for root in roots if root.exists()]
    lookup = build_image_lookup(list(dict.fromkeys(roots)), missing_names)
    resolved = []
    for idx, row in frame.iterrows():
        current = direct_series.loc[idx]
        resolved.append(current or (str(lookup[str(row['image_id'])]) if str(row['image_id']) in lookup else None))
    return pd.Series(resolved, index=frame.index, dtype='object')


df['resolved_image_path'] = resolve_image_paths(df)
missing_images = int(df['resolved_image_path'].isna().sum())
print('Imágenes resueltas:', len(df) - missing_images)
print('Imágenes faltantes:', missing_images)
if missing_images:
    print('Para Colab/Kaggle: subir/agregar images_final y setear DERMAVQA_IMAGE_ROOT o IMAGE_ROOT_OVERRIDE.')
    try:
        display(df.loc[df['resolved_image_path'].isna(), ['split', 'encounter_id', 'image_id', 'image_path']].head(10))
    except NameError:
        print(df.loc[df['resolved_image_path'].isna(), ['split', 'encounter_id', 'image_id', 'image_path']].head(10).to_string())
if missing_images and (RUN_TRAINING or RUN_EVAL_GENERATION):
    raise FileNotFoundError('Faltan imágenes; no se puede entrenar/evaluar un VLM multimodal.')

for image_path in df['resolved_image_path'].dropna().head(3):
    with Image.open(image_path) as image:
        image.verify()
print('OK: muestra de imágenes verificadas.' if missing_images < len(df) else 'Aviso: no hay imágenes para verificar todavía.')


## 3. Formato conversacional multimodal

Cada fila se convierte en conversación `user -> assistant` con un bloque de imagen y otro de texto. El target queda en la respuesta del assistant.


In [ ]:
def clean_text(value: Any) -> str:
    return ' '.join(str(value).split())


def image_uri_from_path(path_value: Any) -> str:
    return Path(str(path_value)).expanduser().resolve().as_uri()


USER_PROMPT_TEMPLATE = 'Sos un asistente de dermatologia. Observa la imagen clinica y responde en espanol con una respuesta breve, medica y fiel al caso.\n\nPregunta: {question_es}'


def build_messages(row: pd.Series | dict[str, Any], include_answer: bool = True) -> list[dict[str, Any]]:
    user_text = USER_PROMPT_TEMPLATE.format(question_es=clean_text(row['question_es']))
    image_content = {'type': 'image'}
    if 'resolved_image_path' in row and clean_text(row['resolved_image_path']):
        image_content['image'] = image_uri_from_path(row['resolved_image_path'])
    messages = [{'role': 'user', 'content': [image_content, {'type': 'text', 'text': user_text}]}]
    if include_answer:
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': clean_text(row['target_answer_es'])}]})
    return messages


df_ready = df.dropna(subset=['resolved_image_path']).copy()
df_ready['question_es'] = df_ready['question_es'].map(clean_text)
df_ready['target_answer_es'] = df_ready['target_answer_es'].map(clean_text)
df_ready['messages'] = df_ready.apply(lambda row: build_messages(row, include_answer=True), axis=1)
print('Filas listas para VLM:', len(df_ready))
print('Filas descartadas por imagen faltante:', len(df) - len(df_ready))
print(json.dumps(df_ready.iloc[0]['messages'], ensure_ascii=False, indent=2) if len(df_ready) else 'Sin ejemplos listos.')


In [ ]:
from datasets import Dataset, DatasetDict

DATASET_COLUMNS = ['split', 'encounter_id', 'image_id', 'resolved_image_path', 'question_es', 'target_answer_es', 'messages']


def split_dataset(frame: pd.DataFrame, split_name: str, max_rows: int | None = None) -> Dataset:
    part = frame.loc[frame['split'] == split_name, DATASET_COLUMNS].reset_index(drop=True)
    if max_rows is not None:
        part = part.head(min(max_rows, len(part)))
    return Dataset.from_pandas(part, preserve_index=False)


train_limit = SMOKE_MAX_TRAIN_SAMPLES if TRAIN_SMOKE_TEST else None
eval_limit = SMOKE_MAX_EVAL_SAMPLES if TRAIN_SMOKE_TEST else None
hf_datasets = DatasetDict({'train': split_dataset(df_ready, 'train', train_limit), 'valid': split_dataset(df_ready, 'valid', eval_limit), 'test': split_dataset(df_ready, 'test', eval_limit)})
print(hf_datasets)


## 4. Modelo Qwen2.5-VL + LoRA/QLoRA

Esta seccion descarga `Qwen/Qwen2.5-VL-3B-Instruct` solo si `RUN_TRAINING=True` o `RUN_EVAL_GENERATION=True`.


In [ ]:
def preferred_torch_dtype():
    import torch
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32


def load_processor_and_model():
    import torch
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    dtype = preferred_torch_dtype()
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        min_pixels=MIN_IMAGE_PIXELS,
        max_pixels=MAX_IMAGE_PIXELS,
    )
    model_kwargs = {'trust_remote_code': True}
    if torch.cuda.is_available():
        model_kwargs['device_map'] = 'auto'
    if USE_QLORA:
        if not torch.cuda.is_available():
            raise RuntimeError('QLoRA 4-bit requiere GPU CUDA. Usa USE_QLORA=False para CPU/local.')
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
    else:
        model_kwargs['torch_dtype'] = dtype
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
    if hasattr(model, 'config'):
        model.config.use_cache = False
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
    if USE_QLORA:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)
    return processor, model


LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = None


def discover_lora_targets(model) -> list[str]:
    import torch
    common = {'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'}
    targets = set()
    for module_name, module in model.named_modules():
        leaf = module_name.split('.')[-1]
        if leaf in common and isinstance(module, torch.nn.Linear):
            targets.add(leaf)
    return sorted(targets) or ['q_proj', 'v_proj']


def attach_lora_adapter(model):
    from peft import LoraConfig, get_peft_model
    target_modules = LORA_TARGET_MODULES or discover_lora_targets(model)
    print('LoRA target modules:', target_modules)
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=target_modules,
    )
    peft_model = get_peft_model(model, peft_config)
    peft_model.print_trainable_parameters()
    return peft_model


if RUN_TRAINING or RUN_EVAL_GENERATION:
    processor, model = load_processor_and_model()
    print('Modelo cargado:', MODEL_ID)
else:
    print('Modelo no cargado. Activa RUN_TRAINING o RUN_EVAL_GENERATION para descargarlo.')

if RUN_TRAINING:
    model = attach_lora_adapter(model)
else:
    print('Adapter LoRA no adjuntado porque RUN_TRAINING=False.')


## 5. Collator y entrenamiento smoke test

El collator aplica el chat template multimodal y enmascara el prompt para entrenar solo la respuesta del assistant.


In [ ]:
def load_rgb_image(path: str) -> Image.Image:
    return Image.open(path).convert('RGB')


def extend_optional(target: list[Any], values: Any) -> None:
    if values is None:
        return
    if isinstance(values, (list, tuple)):
        target.extend(values)
    else:
        target.append(values)


def collect_qwen_vision_inputs(messages_batch: list[list[dict[str, Any]]]) -> tuple[list[Any], list[Any]]:
    from qwen_vl_utils import process_vision_info
    image_inputs: list[Any] = []
    video_inputs: list[Any] = []
    for messages in messages_batch:
        current_images, current_videos = process_vision_info(messages)
        extend_optional(image_inputs, current_images)
        extend_optional(video_inputs, current_videos)
    return image_inputs, video_inputs


def processor_vision_kwargs(messages_batch: list[list[dict[str, Any]]]) -> dict[str, Any]:
    image_inputs, video_inputs = collect_qwen_vision_inputs(messages_batch)
    kwargs: dict[str, Any] = {}
    if image_inputs:
        kwargs['images'] = image_inputs
    if video_inputs:
        kwargs['videos'] = video_inputs
    return kwargs


class VLMDataCollator:
    def __init__(self, processor, max_length: int = 1024):
        self.processor = processor
        self.max_length = max_length
        self.tokenizer = getattr(processor, 'tokenizer', processor)
        self.pad_token_id = getattr(self.tokenizer, 'pad_token_id', None) or getattr(self.tokenizer, 'eos_token_id', 0)

    def chat_text(self, messages: list[dict[str, Any]], add_generation_prompt: bool) -> str:
        return self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)

    def __call__(self, examples: list[dict[str, Any]]) -> dict[str, Any]:
        full_messages = [example['messages'] for example in examples]
        prompt_messages = [example['messages'][:-1] for example in examples]
        full_texts = [self.chat_text(messages, add_generation_prompt=False) for messages in full_messages]
        prompt_texts = [self.chat_text(messages, add_generation_prompt=True) for messages in prompt_messages]
        batch = self.processor(
            text=full_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt',
            **processor_vision_kwargs(full_messages),
        )
        labels = batch['input_ids'].clone()
        labels[labels == self.pad_token_id] = -100
        for row_idx, messages in enumerate(prompt_messages):
            prompt_inputs = self.processor(
                text=[prompt_texts[row_idx]],
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
                **processor_vision_kwargs([messages]),
            )
            prompt_len = int(prompt_inputs['input_ids'].shape[-1])
            labels[row_idx, :min(prompt_len, labels.shape[-1])] = -100
        batch['labels'] = labels
        return batch


def write_run_config() -> Path:
    config = {
        'dataset_variant': DATASET_VARIANT,
        'method': METHOD,
        'model_id': MODEL_ID,
        'use_qlora': USE_QLORA,
        'target_column': TARGET_COLUMN,
        'train_smoke_test': TRAIN_SMOKE_TEST,
        'smoke_max_train_samples': SMOKE_MAX_TRAIN_SAMPLES,
        'smoke_max_eval_samples': SMOKE_MAX_EVAL_SAMPLES,
        'smoke_max_steps': SMOKE_MAX_STEPS,
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'max_text_tokens': MAX_TEXT_TOKENS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'min_image_pixels': MIN_IMAGE_PIXELS,
        'max_image_pixels': MAX_IMAGE_PIXELS,
        'dataset_zip': str(dataset_zip_path),
        'output_dir': str(OUTPUT_DIR),
        'recommended_google_cloud_gpu': 'NVIDIA L4 24GB',
    }
    path = OUTPUT_DIR / 'training_config.json'
    path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    return path


if RUN_TRAINING:
    import torch
    from transformers import Trainer, TrainingArguments
    if len(hf_datasets['train']) == 0:
        raise RuntimeError('No hay filas de train con imagenes resueltas.')
    use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    use_fp16 = bool(torch.cuda.is_available() and not use_bf16)
    training_args = TrainingArguments(
        output_dir=str(ADAPTER_DIR),
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=SMOKE_MAX_STEPS if TRAIN_SMOKE_TEST else -1,
        logging_steps=1,
        save_strategy='no' if TRAIN_SMOKE_TEST else 'epoch',
        eval_strategy='no',
        remove_unused_columns=False,
        report_to='none',
        bf16=use_bf16,
        fp16=use_fp16,
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=hf_datasets['train'], data_collator=VLMDataCollator(processor, max_length=MAX_TEXT_TOKENS))
    trainer.train()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    processor.save_pretrained(ADAPTER_DIR)
    print('Adapter guardado en:', ADAPTER_DIR)
else:
    print('Entrenamiento omitido. Config guardada en:', write_run_config())


## 6. Evaluación valid/test

Activar `RUN_EVAL_GENERATION=True` para crear predicciones. Si existe `adapter/`, se carga sobre el modelo base; si no, se evalúa el base model y se marca en los CSV.


In [ ]:
def first_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        import torch
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def move_batch_to_device(batch: dict[str, Any], device):
    return {key: (value.to(device) if hasattr(value, 'to') else value) for key, value in batch.items()}


def maybe_load_adapter_for_eval():
    global processor, model
    if 'processor' not in globals() or 'model' not in globals():
        processor, model = load_processor_and_model()
    if (ADAPTER_DIR / 'adapter_config.json').exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        print('Adapter cargado:', ADAPTER_DIR)
        return 'adapter'
    print('No encontre adapter_config.json; se usara el modelo base para evaluacion.')
    return 'base_model'


def generate_one(row: pd.Series, processor, model) -> tuple[str, float]:
    import torch
    messages = build_messages(row, include_answer=False)
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[prompt_text],
        padding=True,
        return_tensors='pt',
        **processor_vision_kwargs([messages]),
    )
    inputs = move_batch_to_device(inputs, first_model_device(model))
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    latency_sec = time.perf_counter() - start
    input_len = int(inputs['input_ids'].shape[-1])
    prediction = processor.batch_decode(generated[:, input_len:], skip_special_tokens=True)[0]
    return clean_text(prediction), latency_sec


def prediction_rows_for_split(split_name: str) -> pd.DataFrame:
    split_df = df_ready.loc[df_ready['split'] == split_name].copy()
    if TRAIN_SMOKE_TEST and SMOKE_MAX_EVAL_SAMPLES is not None:
        split_df = split_df.head(min(SMOKE_MAX_EVAL_SAMPLES, len(split_df)))
    rows = []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc='Generating ' + split_name):
        prediction, latency_sec = generate_one(row, processor, model)
        rows.append({
            'split': split_name,
            'encounter_id': row['encounter_id'],
            'image_id': row['image_id'],
            'image_path': row['resolved_image_path'],
            'question_es': row['question_es'],
            'reference_answer_es': row['target_answer_es'],
            'predicted_answer_es': prediction,
            'latency_sec': latency_sec,
            'model_id': MODEL_ID,
            'model_name': MODEL_ID,
            'model_source': EVAL_MODEL_SOURCE,
            'dataset_variant': DATASET_VARIANT,
            'method': METHOD,
        })
    return pd.DataFrame(rows)


if RUN_EVAL_GENERATION:
    EVAL_MODEL_SOURCE = maybe_load_adapter_for_eval()
    model.eval()
    predictions_by_split = {}
    for split_name in ['valid', 'test']:
        pred_df = prediction_rows_for_split(split_name)
        path = OUTPUT_DIR / ('predictions_' + split_name + '.csv')
        pred_df.to_csv(path, index=False, encoding='utf-8')
        predictions_by_split[split_name] = pred_df
        print('Predicciones guardadas:', path, 'filas:', len(pred_df))
else:
    EVAL_MODEL_SOURCE = 'not_run'
    predictions_by_split = {}
    print('Generacion omitida. Activa RUN_EVAL_GENERATION=True para crear predictions_valid/test.csv.')


## 7. Métricas comunes

Calcula sacreBLEU, chrF, ROUGE-L, token F1, latencia promedio/p50/p95 y BERTScore opcional.


In [ ]:
def token_f1(prediction: str, reference: str) -> float:
    pred_tokens = clean_text(prediction).lower().split()
    ref_tokens = clean_text(reference).lower().split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    ref_counts = {}
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1
    overlap = 0
    for token in pred_tokens:
        if ref_counts.get(token, 0) > 0:
            overlap += 1
            ref_counts[token] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def lcs_length(left: list[str], right: list[str]) -> int:
    previous = [0] * (len(right) + 1)
    for left_token in left:
        current = [0]
        for j, right_token in enumerate(right, start=1):
            current.append(previous[j - 1] + 1 if left_token == right_token else max(previous[j], current[-1]))
        previous = current
    return previous[-1]


def rouge_l_f1(prediction: str, reference: str) -> float:
    pred_tokens = clean_text(prediction).lower().split()
    ref_tokens = clean_text(reference).lower().split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    lcs = lcs_length(pred_tokens, ref_tokens)
    if lcs == 0:
        return 0.0
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_metrics(pred_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    import sacrebleu
    predictions = pred_df['predicted_answer_es'].fillna('').astype(str).tolist()
    references = pred_df['reference_answer_es'].fillna('').astype(str).tolist()
    latency = pred_df['latency_sec'].astype(float) if 'latency_sec' in pred_df else pd.Series(dtype=float)
    row = {'split': split_name, 'dataset_variant': DATASET_VARIANT, 'method': METHOD, 'model_id': MODEL_ID, 'model_name': MODEL_ID, 'model_source': pred_df['model_source'].iloc[0] if len(pred_df) else EVAL_MODEL_SOURCE, 'n_examples': len(pred_df), 'sacrebleu': sacrebleu.corpus_bleu(predictions, [references]).score, 'chrf': sacrebleu.corpus_chrf(predictions, [references]).score, 'rouge_l_f1': float(np.mean([rouge_l_f1(pred, ref) for pred, ref in zip(predictions, references)])) if predictions else np.nan, 'token_f1': float(np.mean([token_f1(pred, ref) for pred, ref in zip(predictions, references)])) if predictions else np.nan, 'latency_mean_sec': float(latency.mean()) if len(latency) else np.nan, 'latency_p50_sec': float(latency.quantile(0.50)) if len(latency) else np.nan, 'latency_p95_sec': float(latency.quantile(0.95)) if len(latency) else np.nan}
    if RUN_BERTSCORE:
        from bert_score import score
        precision, recall, f1 = score(predictions, references, lang='es', verbose=True, rescale_with_baseline=False)
        row.update({'bertscore_precision': float(precision.mean()), 'bertscore_recall': float(recall.mean()), 'bertscore_f1': float(f1.mean())})
    return pd.DataFrame([row])


metrics_by_split = {}
if RUN_EVAL_GENERATION:
    for split_name, pred_df in predictions_by_split.items():
        metrics_df = compute_metrics(pred_df, split_name)
        metrics_path = OUTPUT_DIR / ('metrics_' + split_name + '.csv')
        metrics_df.to_csv(metrics_path, index=False, encoding='utf-8')
        metrics_by_split[split_name] = metrics_df
        print('Métricas guardadas:', metrics_path)
        try:
            display(metrics_df)
        except NameError:
            print(metrics_df.to_string(index=False))
else:
    print('Métricas omitidas porque no se generaron predicciones.')


## 8. Revisión manual y zip final

El zip final incluye adapter si existe, configuración, predicciones, métricas y `manual_review_20.csv`.


In [ ]:
def build_manual_review(predictions: dict[str, pd.DataFrame], sample_size: int = 20) -> pd.DataFrame:
    available = [frame for frame in predictions.values() if frame is not None and len(frame)]
    if not available:
        return pd.DataFrame()
    combined = pd.concat(available, ignore_index=True)
    review = combined.sample(min(sample_size, len(combined)), random_state=RANDOM_SEED).copy()
    review.insert(0, 'review_status', '')
    review.insert(1, 'review_notes', '')
    keep = ['review_status', 'review_notes', 'split', 'encounter_id', 'image_id', 'image_path', 'question_es', 'reference_answer_es', 'predicted_answer_es', 'latency_sec', 'model_id', 'model_name', 'model_source', 'dataset_variant', 'method']
    return review[keep]


if RUN_EVAL_GENERATION:
    manual_review_df = build_manual_review(predictions_by_split, sample_size=20)
    manual_review_path = OUTPUT_DIR / 'manual_review_20.csv'
    manual_review_df.to_csv(manual_review_path, index=False, encoding='utf-8')
    print('Revisión manual guardada:', manual_review_path)
else:
    print('manual_review_20.csv se genera después de RUN_EVAL_GENERATION=True.')


def add_file_to_zip(archive: zipfile.ZipFile, path: Path, arcname: str | None = None) -> None:
    if path.exists() and path.is_file():
        archive.write(path, arcname=arcname or path.name)


def package_outputs() -> Path:
    archive_path = OUTPUT_DIR / 'dataset_enriched_vlm_lora_adapter_results.zip'
    if archive_path.exists():
        archive_path.unlink()
    expected_files = ['README.md', 'training_config.json', 'predictions_valid.csv', 'predictions_test.csv', 'metrics_valid.csv', 'metrics_test.csv', 'manual_review_20.csv']
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for file_name in expected_files:
            add_file_to_zip(archive, OUTPUT_DIR / file_name, arcname=file_name)
        if ADAPTER_DIR.exists():
            for path in ADAPTER_DIR.rglob('*'):
                if path.is_file():
                    archive.write(path, arcname=str(Path('adapter') / path.relative_to(ADAPTER_DIR)))
    return archive_path


if PACKAGE_RESULTS:
    archive_path = package_outputs()
    print('Zip final:', archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        for name in archive.namelist():
            print('-', name)
    try:
        from google.colab import files  # type: ignore
        files.download(str(archive_path))
    except Exception:
        print('Descargá o versioná el zip desde:', archive_path)
else:
    print('Packaging omitido. Activá PACKAGE_RESULTS=True cuando existan outputs reales.')


## 9. Checklist de entrega

1. Crear budget alert en Google Cloud y confirmar que la VM L4 tiene GPU con `nvidia-smi`.
2. Validar conteos, splits, leakage e imagenes con `DERMAVQA_IMAGE_ROOT` apuntando a `images_final`.
3. Activar `RUN_TRAINING=True` con `TRAIN_SMOKE_TEST=True` para smoke test.
4. Si no hay OOM, correr `TRAIN_SMOKE_TEST=False` y `NUM_TRAIN_EPOCHS=3`.
5. Activar `RUN_EVAL_GENERATION=True` para escribir `predictions_valid.csv` y `predictions_test.csv`.
6. Revisar `metrics_valid.csv`, `metrics_test.csv` y `manual_review_20.csv`.
7. Activar `PACKAGE_RESULTS=True` y verificar `dataset_enriched_vlm_lora_adapter_results.zip`.
8. Apagar la VM al terminar para cuidar los creditos.
